In [0]:
from pyspark.sql import functions as F

source = "/Volumes/workspace/default/raw/salesforce/accounts/"
checkpoint = "/Volumes/workspace/default/raw/_checkpoints/bronze_accounts"

q = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.useIncrementalListing", "false")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", checkpoint)
        .option("pathGlobFilter", "*.json")
        .load(source)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    .writeStream
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable("workspace.bronze.salesforce_accounts")
    )

q.awaitTermination()

In [0]:
%sql
SELECT
    COUNT(*)            AS total_rows
  , COUNT(DISTINCT id)  AS distinct_accounts
FROM
    workspace.bronze.salesforce_accounts